# Libraries

In [1]:
# Корень репозитория (где pyproject.toml) → в sys.path добавляется src/
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    _src = _p / "src"
    if (_p / "pyproject.toml").is_file() and _src.is_dir():
        _s = str(_src)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break

import json
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from models.cnn_encoder import RawCNNClassifier
from metrics.metrics import compute_classification_metrics
from data import load_dataset


from statistical.quantile_extractor import (
    STAT_METHODS_GLOBAL_TORCH,
    STAT_METHODS_TORCH,
    TorchQuantileExtractor,
)

# Data

In [2]:
basic_motions_dataset = load_dataset("BasicMotions")


In [3]:
print(basic_motions_dataset.X_train.shape)
print(basic_motions_dataset.y_train.shape)
print(basic_motions_dataset.X_test.shape)
print(basic_motions_dataset.y_test.shape)


(40, 6, 100)
(40,)
(40, 6, 100)
(40,)


In [4]:
ford_a_dataset = load_dataset("FordA")
print(ford_a_dataset.X_train.shape)
print(ford_a_dataset.y_train.shape)
print(ford_a_dataset.X_test.shape)
print(ford_a_dataset.y_test.shape)


(3601, 1, 500)
(3601,)
(1320, 1, 500)
(1320,)


# Tools

In [5]:
def save_result(result, name):
    with open(f"{name}.json", "w") as f:
        json.dump(result, f)

# Simple CNN Encoder

In [6]:
dataset_name = "FordA"
ds = ford_a_dataset

CLASS_NAMES = {
    "BasicMotions": ["walking", "resting", "running", "badminton"],
    "FordA": ["FordA_-1", "FordA_1"],
}

batch_size = min(32, len(ds.X_train))
channels = ds.X_train.shape[1]
time_steps = ds.X_train.shape[2]
num_classes = int(ds.y_train.max()) + 1
class_names = CLASS_NAMES.get(dataset_name, [f"class_{i}" for i in range(num_classes)])

n_epochs = 100
lr = 1e-3
seed = 42

In [7]:
def set_seed(s: int) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed(seed)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

X_train_t = torch.from_numpy(ds.X_train).float()
y_train_t = torch.from_numpy(ds.y_train).long()
X_test_t = torch.from_numpy(ds.X_test).float()
y_test_t = torch.from_numpy(ds.y_test).long()

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=batch_size,
    shuffle=False,
)

model = RawCNNClassifier(
    in_channels=channels,
    num_classes=num_classes,
    d_model=128,
    hidden_channels=(64, 128, 128),
    kernel_size=5,
    encoder_dropout=0.1,
    head_dropout=0.2,
).to(device)

xb, yb = next(iter(train_loader))
with torch.no_grad():
    print("logits shape:", model(xb.to(device)).shape)

logits shape: torch.Size([32, 2])


In [8]:
device

device(type='mps')

In [9]:
def train_raw_cnn(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    *,
    n_epochs: int,
    lr: float,
) -> list[float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses: list[float] = []

    for _ in tqdm(range(n_epochs)):
        running = 0.0
        n_seen = 0
        for x_raw, y in train_loader:
            x_raw = x_raw.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_raw)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            bs = y.size(0)
            running += loss.item() * bs
            n_seen += bs
        epoch_losses.append(running / max(n_seen, 1))

    return epoch_losses

In [10]:
epoch_losses = train_raw_cnn(
    model,
    train_loader,
    device,
    n_epochs=n_epochs,
    lr=lr,
)
print("last train loss:", epoch_losses[-1])

100%|██████████| 100/100 [01:43<00:00,  1.03s/it]

last train loss: 0.15793118997477254


In [12]:
@torch.no_grad()
def evaluate_model(
    model,
    dataloader,
    device,
    class_names=None,
):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    for batch in dataloader:
        x_raw, y = batch

        x_raw = x_raw.to(device)
        y = y.to(device)

        logits = model(x_raw)
        loss = criterion(logits, y)

        preds = torch.argmax(logits, dim=1)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_samples += bs

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        class_names=class_names,
    )

    metrics["loss"] = total_loss / total_samples

    return metrics

In [13]:
metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=class_names,
)

print("Loss:", metrics["loss"])
print("Accuracy:", metrics["accuracy"])
print("Macro F1:", metrics["macro_f1"])
print("Weighted F1:", metrics["weighted_f1"])

print("Confusion matrix:")
print(metrics["confusion_matrix"])

print("Classification report:")
print(metrics["classification_report"])

Loss: 0.1982240783671538
Accuracy: 0.9272727272727272
Macro F1: 0.9271119250688502
Weighted F1: 0.9272208555940572
Confusion matrix:
[[643  38]
 [ 58 581]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.92      0.94      0.93       681
     FordA_1       0.94      0.91      0.92       639

    accuracy                           0.93      1320
   macro avg       0.93      0.93      0.93      1320
weighted avg       0.93      0.93      0.93      1320



In [16]:
encoder_result = {
    "experiment_name": "E1_raw_cnn_baseline",
    "modalities": ["raw"],
    "fusion": None,
    "encoder": "1D CNN",
    "loss": float(metrics["loss"]),
    "accuracy": float(metrics["accuracy"]),
    "macro_f1": float(metrics["macro_f1"]),
    "weighted_f1": float(metrics["weighted_f1"]),
    "confusion_matrix": np.asarray(metrics["confusion_matrix"]).tolist(),
}
save_result(encoder_result, "results/experiments/E1_raw_cnn_baseline_FordA")

# Encoder for statistical features

## Статистические признаки (FordA): TorchQuantileExtractor

FordA: форма временного ряда (B, 1, T) с T=500. **TorchQuantileExtractor нужно кормить torch.Tensor**; для этого пайплайна удобно брать **(B, T)**, например x.squeeze(1) при одном канале: при **stride > 1** включается матрица Ганкеля по размерности времени — для данных вида **(B, 1, T)** результат получается неверным, для **`(B, T)`** — ожидаемый.

**Параметры под последующий энкодер / классификацию:**
- **STAT_PARAMS_COMPACT** — только глобальные + статистики по всему ряду (window_size=0): **фиксированный короткий вектор (28 признаков)**, удобная размерность для MLP без «раздувания» входа и без лишнего переобучения на малых обучающих выборках типа UCR.
- **STAT_PARAMS_WINDOWS** — дополнительно скользящие окна: window_size=12 интерпретируется как **~12 % от T** (см. apply_window_for_stat_feature_torch), **stride=50** уменьшает число окон и размерность; при **T=500** типичная форма признаков — **(..., 100)** (**19 глобальных + 9 окон × 9 локальных** при выбранных числах; точное число окон см. вывод следующей ячейки).

Ниже: размерность train/test после извлечения и пример среза вектора для одного образца.

In [5]:
def ford_a_feat_tensor(X: torch.Tensor) -> torch.Tensor:
    """FordA имеет один канал: (B, 1, T) → (B, T) для стабильного окна/Hankel по времени."""
    X = X.float()
    if X.ndim == 3 and X.shape[1] == 1:
        return X.squeeze(1)
    return X


STAT_PARAMS_COMPACT = {
    "window_size": 0,
    "stride": 1,
    "add_global_features": True,
}

STAT_PARAMS_WINDOWS = {
    "window_size": 12,
    "stride": 50,
    "add_global_features": True,
}


def feats_batched(
    extractor: TorchQuantileExtractor,
    X_2d: torch.Tensor,
    batch_size: int = 512,
) -> torch.Tensor:
    chunks: list[torch.Tensor] = []
    for start in range(0, len(X_2d), batch_size):
        chunk = X_2d[start:start + batch_size]
        chunks.append(extractor.generate_features_from_ts(chunk))
    return torch.cat(chunks, dim=0)


X_train_2d = ford_a_feat_tensor(torch.from_numpy(ford_a_dataset.X_train))
X_test_2d = ford_a_feat_tensor(torch.from_numpy(ford_a_dataset.X_test))

configs = (
    ("COMPACT → энкодер по умолчанию", STAT_PARAMS_COMPACT),
    ("WINDOWS → больше локальной информации", STAT_PARAMS_WINDOWS),
)

for label, stat_params in configs:
    extractor = TorchQuantileExtractor(stat_params)
    F_train = feats_batched(extractor, X_train_2d)
    F_test = feats_batched(extractor, X_test_2d)
    print(label)
    print("  параметры:", stat_params)
    print("  признаки train:", tuple(F_train.shape))
    print("  признаки test: ", tuple(F_test.shape))
    print(
        "  пример строки train[0, :12]:",
        F_train[0, :12].detach().tolist(),
    )
    print()


print(
    "Если window_size=0 — порядок в векторе: первые "
    f"{len(STAT_METHODS_GLOBAL_TORCH)} глобальных, затем {len(STAT_METHODS_TORCH)} статистик по всему ряду."
)
print("Глобальные имена:")
print(" ", ", ".join(STAT_METHODS_GLOBAL_TORCH.keys()))
print("По-серии (локальные) имена:")
print(" ", ", ".join(STAT_METHODS_TORCH.keys()))

COMPACT → энкодер по умолчанию
  параметры: {'window_size': 0, 'stride': 1, 'add_global_features': True}
  признаки train: (3601, 28)
  признаки test:  (1320, 28)
  пример строки train[0, :12]: [0.07592615485191345, -0.8642673492431641, 27.0, 2.8302007194724865e-05, 0.8931005597114563, 1.4827055931091309, 0.9980000257492065, 0.07199999690055847, 0.9695836305618286, 8.664726257324219, 4.462869644165039, 18.423076629638672]

WINDOWS → больше локальной информации
  параметры: {'window_size': 12, 'stride': 50, 'add_global_features': True}
  признаки train: (3601, 100)
  признаки test:  (1320, 100)
  пример строки train[0, :12]: [0.07592615485191345, -0.8642673492431641, 27.0, 2.8302007194724865e-05, 0.8931005597114563, 1.4827055931091309, 0.9980000257492065, 0.07199999690055847, 0.9695836305618286, 8.664726257324219, 4.462869644165039, 18.423076629638672]

Если window_size=0 — порядок в векторе: первые 19 глобальных, затем 9 статистик по всему ряду.
Глобальные имена:
  skewness_, kurtosis_